# Notebook 01 — Pose Extraction Exploration
**Owner: Member 1**

Verify that `PoseEstimator` produces correct keypoints on sample dance clips.
Check confidence distributions and visualize skeleton overlays.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import cv2
import matplotlib.pyplot as plt
from video_io import load_video, standardize
from pose_extraction import PoseEstimator, smooth, interpolate_missing

In [ ]:
# TODO: set this to one of your test clips
VIDEO_PATH = '../data/phrase_01/benchmark.mp4'

frames_raw, src_fps = load_video(VIDEO_PATH)
frames, fps = standardize(frames_raw, src_fps, target_fps=15)
print(f'Loaded {len(frames)} frames at {fps} fps')

In [ ]:
estimator = PoseEstimator(backend='mediapipe')
kp_seq = estimator.extract_sequence(frames)
estimator.close()
print('Raw keypoint sequence shape:', kp_seq.shape)  # (T, 17, 3)

In [ ]:
# Confidence histogram — all joints across all frames
confidences = kp_seq[:, :, 2].flatten()
plt.hist(confidences, bins=30)
plt.axvline(0.3, color='red', linestyle='--', label='threshold=0.3')
plt.xlabel('Confidence')
plt.ylabel('Count')
plt.title('Joint confidence distribution')
plt.legend()
plt.show()
print(f'Frames with any low-confidence joint (<0.3): {(kp_seq[:,:,2] < 0.3).any(axis=1).sum()}')

In [ ]:
# Apply smoothing + interpolation
kp_clean = interpolate_missing(smooth(kp_seq))
print('Cleaned keypoint sequence shape:', kp_clean.shape)

In [ ]:
# Visualize skeleton on a single frame
from visualization import overlay_skeleton

t = 10  # frame index to inspect
annotated = overlay_skeleton(frames[t], kp_clean[t])
plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'Frame {t} — skeleton overlay')
plt.show()